# LeetCode 2055 — Plates Between Candles bằng Python

Link đề: https://leetcode.com/problems/plates-between-candles/ · Độ khó: **Medium** · Chủ đề: Array, String, Binary Search, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def platesBetweenCandles(self, s: str, queries: List[List[int]]) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Nhận ra "đĩa nằm giữa hai nến trong đoạn `[l, r]`" ⇔ "đĩa nằm giữa **nến đầu tiên ≥ l** và **nến cuối cùng ≤ r**".
2. Tiền xử lý `O(n)` ba mảng: **tổng tiền tố số đĩa**, **nến gần nhất bên phải** (`next_candle`), **nến gần nhất bên trái** (`prev_candle`) ⇒ mỗi truy vấn `O(1)`.
3. Cách thay thế: lưu **vị trí các nến** và dùng `bisect` — số đĩa giữa nến `i` và nến `j` là `(pos[j] − pos[i]) − (j − i)`.
4. Xử lý trường hợp đoạn có **ít hơn hai nến** (đáp án 0, không âm).
5. Phân biệt rõ `O(n·q)` (quét từng truy vấn) với `O(n + q)`.

## 2. Đề bài & ví dụ

Chuỗi `s` gồm `'*'` (đĩa) và `'|'` (nến). Mỗi truy vấn `[left, right]` xét chuỗi con `s[left..right]` (bao gồm). Đếm số **đĩa nằm giữa nến** trong chuỗi con: đĩa có ít nhất một nến **bên trái** và ít nhất một nến **bên phải** **trong chuỗi con**.

Ví dụ: `s = "||**||**|*"`, truy vấn `[3, 8]` ⇒ chuỗi con `"*||**|"` có 2 đĩa giữa nến.

**Ví dụ 1:** `s = "**|**|***|"`, `queries = [[2,5],[5,9]]` → `[2,3]`.

**Ví dụ 2:** `s = "***|**|*****|**||**|*"`, `queries = [[1,17],[4,5],[14,17],[5,11],[15,16]]` → `[9,0,0,0,0]`.

**Ràng buộc:** `3 ≤ n ≤ 10⁵`; `1 ≤ q ≤ 10⁵`; `0 ≤ left ≤ right < n`.

In [1]:
import random
import time
import bisect
from itertools import accumulate
from typing import List

examples = [(("**|**|***|", [[2, 5], [5, 9]]), [2, 3]),
            (("***|**|*****|**||**|*", [[1, 17], [4, 5], [14, 17], [5, 11], [15, 16]]), [9, 0, 0, 0, 0])]
for (s, qs), exp in examples:
    print(f"s={s!r}, queries={qs} -> kỳ vọng {exp}")

s='**|**|***|', queries=[[2, 5], [5, 9]] -> kỳ vọng [2, 3]
s='***|**|*****|**||**|*', queries=[[1, 17], [4, 5], [14, 17], [5, 11], [15, 16]] -> kỳ vọng [9, 0, 0, 0, 0]


## 3. Kiến thức nền

### 3.1 Tổng tiền tố đếm ký tự

`pre[i]` = số `'*'` trong `s[0..i−1]` ⇒ số đĩa trong `s[a..b]` là `pre[b+1] − pre[a]`.

In [2]:
s = "**|**|***|"
pre = list(accumulate((c == "*" for c in s), initial=0))
print("pre =", pre, "| số đĩa trong s[2..5] =", pre[6] - pre[2])

pre = [0, 1, 2, 2, 3, 4, 4, 5, 6, 7, 7] | số đĩa trong s[2..5] = 2


### 3.2 Phần tử gần nhất bên phải / trái thỏa điều kiện

`nxt[i]` = chỉ số nến nhỏ nhất `≥ i` (hoặc `∞`), tính từ phải sang trái; `prv[i]` = chỉ số nến lớn nhất `≤ i` (hoặc `−1`), tính từ trái sang phải. Mỗi mảng `O(n)`.

In [3]:
def near_candles(s):
    n = len(s); nxt = [n] * (n + 1); prv = [-1] * n
    for i in range(n - 1, -1, -1):
        nxt[i] = i if s[i] == "|" else nxt[i + 1]
    last = -1
    for i in range(n):
        if s[i] == "|": last = i
        prv[i] = last
    return nxt[:n], prv

nxt, prv = near_candles(s)
print("s   =", list(s)); print("nxt =", nxt); print("prv =", prv)

s   = ['*', '*', '|', '*', '*', '|', '*', '*', '*', '|']
nxt = [2, 2, 2, 5, 5, 5, 9, 9, 9, 9]
prv = [-1, -1, 2, 2, 2, 5, 5, 5, 5, 9]


### 3.3 `bisect` trên danh sách vị trí

Với `pos` = danh sách vị trí nến (tăng dần): nến đầu tiên `≥ l` có chỉ số `bisect_left(pos, l)`, nến cuối cùng `≤ r` có chỉ số `bisect_right(pos, r) − 1`.

### 3.4 Số phần tử "không phải nến" giữa hai nến

Giữa nến thứ `i` (vị trí `pos[i]`) và nến thứ `j` (`pos[j]`), có `pos[j] − pos[i] − 1` ký tự, trong đó `j − i − 1` là nến ⇒ số đĩa `= pos[j] − pos[i] − (j − i)`.

## 4. Ý tưởng nền tảng / trực giác

Trong `s[l..r]`, gọi `a` = nến **đầu tiên** ở vị trí `≥ l`, `b` = nến **cuối cùng** ở vị trí `≤ r`. Một đĩa ở vị trí `x ∈ [l, r]` có nến bên trái và bên phải trong đoạn ⇔ `a < x < b`. Vậy:

- Nếu `a < b`: đáp án = số đĩa trong `s[a..b]` = `pre[b+1] − pre[a]` (hai đầu là nến nên không ảnh hưởng).
- Ngược lại (0 hoặc 1 nến trong đoạn): đáp án `0`.

Tính tay **Ví dụ 1** `s = "**|**|***|"` (nến ở 2, 5, 9):

| truy vấn | a (nến ≥ l) | b (nến ≤ r) | đĩa trong `s[a..b]` |
|---|---|---|---|
| `[2, 5]` | 2 | 5 | `s[3..4] = "**"` ⇒ 2 |
| `[5, 9]` | 5 | 9 | `s[6..8] = "***"` ⇒ 3 |

In [4]:
for l, r in [[2, 5], [5, 9], [0, 1], [3, 4]]:
    a, b = nxt[l], prv[r]
    ans = pre[b + 1] - pre[a] if a < b else 0
    print(f"[{l},{r}] ({s[l:r + 1]!r}): a={a}, b={b} -> {ans}")

[2,5] ('|**|'): a=2, b=5 -> 2
[5,9] ('|***|'): a=5, b=9 -> 3
[0,1] ('**'): a=2, b=-1 -> 0
[3,4] ('**'): a=5, b=2 -> 0


## 5. Cách 1 — Brute Force / baseline: quét từng truy vấn

Trong `s[l..r]`, tìm nến đầu và nến cuối bằng `find`/`rfind`, rồi đếm `'*'` ở giữa.

In [5]:
def plates_bruteforce(s, queries):
    out = []
    for l, r in queries:
        sub = s[l:r + 1]
        a, b = sub.find("|"), sub.rfind("|")
        out.append(sub[a:b].count("*") if a != -1 and a < b else 0)
    return out

for (s_, qs), exp in examples:
    got = plates_bruteforce(s_, qs); print(got, "kỳ vọng", exp); assert got == exp

[2, 3] kỳ vọng [2, 3]
[9, 0, 0, 0, 0] kỳ vọng [9, 0, 0, 0, 0]


**Độ phức tạp:** `O(n)` mỗi truy vấn ⇒ `O(n·q) = 10¹⁰` tệ nhất ⇒ TLE (dù các thao tác chuỗi chạy bằng C).

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (thu hẹp về hai nến):** chỉ đĩa nằm giữa nến đầu `a` và nến cuối `b` của đoạn mới được tính.

**Insight 2 (tiền xử lý):** `a = nxt[l]`, `b = prv[r]`, số đĩa `pre[b+1] − pre[a]` — tất cả `O(1)` sau `O(n)` tiền xử lý.

**Insight 3 (chỉ cần vị trí nến):** với `pos` và `bisect`, `i = bisect_left(pos, l)`, `j = bisect_right(pos, r) − 1`; nếu `i < j` đáp án `pos[j] − pos[i] − (j − i)`.

**Invariant:** `a ≥ l` và `b ≤ r` theo định nghĩa ⇒ `[a, b] ⊆ [l, r]` khi `a < b`.

### 6.1 Minh họa: điều kiện `a < b` bắt mọi trường hợp "không đủ nến"

- Đoạn **không có nến**: `nxt[l] > r` và `prv[r] < l` ⇒ `a > b`.
- Đoạn **đúng một nến** ở `c`: `a = b = c`.

In [6]:
s2 = "***|**|*****|**||**|*"
nxt2, prv2 = near_candles(s2)
for l, r in [[4, 5], [14, 17], [15, 16], [0, 2], [3, 3]]:
    print(f"[{l},{r}] {s2[l:r + 1]!r}: a={nxt2[l]}, b={prv2[r]} -> {'a < b' if nxt2[l] < prv2[r] else 'không đủ hai nến -> 0'}")

[4,5] '**': a=6, b=3 -> không đủ hai nến -> 0
[14,17] '*||*': a=15, b=16 -> a < b
[15,16] '||': a=15, b=16 -> a < b
[0,2] '***': a=3, b=-1 -> không đủ hai nến -> 0
[3,3] '|': a=3, b=3 -> không đủ hai nến -> 0


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tiền tố + nến gần nhất (bản nộp)

In [7]:
def plates_prefix(s, queries):
    n = len(s)
    pre = [0] * (n + 1)
    for i, c in enumerate(s):
        pre[i + 1] = pre[i] + (c == "*")
    nxt = [n] * (n + 1)
    for i in range(n - 1, -1, -1):
        nxt[i] = i if s[i] == "|" else nxt[i + 1]
    prv = [-1] * n
    last = -1
    for i, c in enumerate(s):
        if c == "|":
            last = i
        prv[i] = last
    out = []
    for l, r in queries:
        a, b = nxt[l], prv[r]
        out.append(pre[b + 1] - pre[a] if a < b else 0)
    return out

print([plates_prefix(*a) for a, _ in examples])

[[2, 3], [9, 0, 0, 0, 0]]


**Độ phức tạp Cách 2:** `O(n + q)` thời gian, `O(n)` bộ nhớ.

### 7.2 Cách 3 — Vị trí nến + `bisect`

In [8]:
def plates_bisect(s, queries):
    pos = [i for i, c in enumerate(s) if c == "|"]
    out = []
    for l, r in queries:
        i = bisect.bisect_left(pos, l)
        j = bisect.bisect_right(pos, r) - 1
        out.append(pos[j] - pos[i] - (j - i) if i < j else 0)
    return out

print([plates_bisect(*a) for a, _ in examples])

[[2, 3], [9, 0, 0, 0, 0]]


**Độ phức tạp Cách 3:** `O(n + q log n)`; bộ nhớ `O(số nến)`.

### 7.3 Cách 4 — Đếm "đĩa đã thấy giữa nến" theo vị trí nến

Một biến thể của Cách 2: `cum[i]` = số đĩa nằm **trước nến gần nhất `≤ i`**. Khi đó đáp án = `cum[b] − cum[a]` với `a`, `b` như trên. Không cần mảng tiền tố riêng của đĩa.

In [9]:
class Solution:
    def platesBetweenCandles(self, s: str, queries: List[List[int]]) -> List[int]:
        n = len(s)
        left = [-1] * n                   # nến gần nhất <= i
        cum = [0] * n                     # số đĩa trước nến left[i]
        plates, last = 0, -1
        for i, c in enumerate(s):
            if c == "|":
                last = i
                cnt_at_last = plates
            else:
                plates += 1
            left[i] = last
            cum[i] = cnt_at_last if last != -1 else 0
        right = [n] * n                   # nến gần nhất >= i
        nxt = n
        for i in range(n - 1, -1, -1):
            if s[i] == "|":
                nxt = i
            right[i] = nxt
        res = []
        for l, r in queries:
            a, b = right[l], left[r]
            res.append(cum[b] - cum[a] if a < b else 0)
        return res

print([Solution().platesBetweenCandles(*a) for a, _ in examples])

[[2, 3], [9, 0, 0, 0, 0]]


`cum[b]` (với `b` là nến) = số đĩa trước `b`; `cum[a]` = số đĩa trước `a` ⇒ hiệu là số đĩa giữa hai nến.

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 2**, truy vấn `[1, 17]`:

In [10]:
s2 = "***|**|*****|**||**|*"
pre2 = list(accumulate((c == "*" for c in s2), initial=0)); nxt2, prv2 = near_candles(s2)
l, r = 1, 17
a, b = nxt2[l], prv2[r]
print("s[1..17] =", repr(s2[l:r + 1]))
print(f"nến đầu >= {l}: a = {a} | nến cuối <= {r}: b = {b}")
print(f"đĩa trong s[{a}..{b}] = pre[{b + 1}] - pre[{a}] = {pre2[b + 1]} - {pre2[a]} = {pre2[b + 1] - pre2[a]}")
print("kiểm tra:", s2[a:b + 1], "->", s2[a:b + 1].count("*"))

s[1..17] = '**|**|*****|**||*'
nến đầu >= 1: a = 3 | nến cuối <= 17: b = 16
đĩa trong s[3..16] = pre[17] - pre[3] = 12 - 3 = 9
kiểm tra: |**|*****|**|| -> 9


| bước | giá trị |
|---|---|
| nến trong `s` | 3, 6, 12, 15, 16, 19 |
| `a = nxt[1]` | 3 |
| `b = prv[17]` | 16 |
| `s[3..16]` | `"\|**\|*****\|**\|\|"` |
| số đĩa | `2 + 5 + 2 = 9` |

Truy vấn `[14, 17]` (`"*||*"`): `a = 15`, `b = 16` ⇒ không có đĩa giữa ⇒ 0. Truy vấn `[5, 11]`: `a = 6`, `b = 6` ⇒ chỉ một nến ⇒ 0.

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — tiền tố đĩa + nến gần nhất hai phía.

In [11]:
class Solution:
    def platesBetweenCandles(self, s: str, queries: List[List[int]]) -> List[int]:
        n = len(s)
        pre = [0] * (n + 1)
        for i, c in enumerate(s):
            pre[i + 1] = pre[i] + (c == "*")
        nxt = [n] * (n + 1)
        for i in range(n - 1, -1, -1):
            nxt[i] = i if s[i] == "|" else nxt[i + 1]
        prv = [-1] * n
        last = -1
        for i, c in enumerate(s):
            if c == "|":
                last = i
            prv[i] = last
        res = []
        for l, r in queries:
            a, b = nxt[l], prv[r]
            res.append(pre[b + 1] - pre[a] if a < b else 0)
        return res

for (s_, qs), exp in examples:
    got = Solution().platesBetweenCandles(s_, qs); print(got, exp); assert got == exp

[2, 3] [2, 3]
[9, 0, 0, 0, 0] [9, 0, 0, 0, 0]


## 10. Vì sao thuật toán đúng

Xét truy vấn `[l, r]`, `a = nxt[l]` (nến nhỏ nhất `≥ l`), `b = prv[r]` (nến lớn nhất `≤ r`).

- Đĩa `x ∈ [l, r]` được tính ⇔ có nến `c₁ ∈ [l, x)` và `c₂ ∈ (x, r]`.
- (⇒) Nếu có `c₁` thì `a ≤ c₁ < x`; nếu có `c₂` thì `x < c₂ ≤ b`. Vậy `a < x < b`.
- (⇐) Nếu `a < x < b` thì `a ≥ l` và `b ≤ r` là các nến trong đoạn ở hai phía của `x`.

Vậy tập đĩa cần đếm = các đĩa ở vị trí `(a, b)` khi `a < b`, rỗng khi `a ≥ b` (0 hoặc 1 nến trong đoạn, hoặc `a > r`). Số đĩa ở `(a, b)` = số đĩa ở `[a, b]` (hai đầu là nến) = `pre[b+1] − pre[a]`. ∎

### 10.1 Kiểm chứng thực nghiệm theo định nghĩa từng đĩa

Viết một bản đếm **đúng từng chữ của đề** (với mỗi đĩa, kiểm tra có nến trái và phải trong đoạn) và so với các cách trên mọi truy vấn của chuỗi ngắn.

In [12]:
def by_definition(s, l, r):
    cnt = 0
    for x in range(l, r + 1):
        if s[x] == "*" and "|" in s[l:x] and "|" in s[x + 1:r + 1]:
            cnt += 1
    return cnt

rng = random.Random(1010)
total = 0
for _ in range(300):
    n = rng.randint(3, 14); s_ = "".join(rng.choice("*|") for _ in range(n))
    qs = [[l, r] for l in range(n) for r in range(l, n)]
    ref = [by_definition(s_, l, r) for l, r in qs]
    assert ref == plates_bruteforce(s_, qs) == plates_prefix(s_, qs) == plates_bisect(s_, qs) == Solution().platesBetweenCandles(s_, qs)
    total += len(qs)
print(f"Định nghĩa = 4 cách trên {total} truy vấn (mọi đoạn của 300 chuỗi)")

Định nghĩa = 4 cách trên 13861 truy vấn (mọi đoạn của 300 chuỗi)


## 11. Phân tích Time / Space Complexity

Gọi `n, q ≤ 10⁵`.

| Cách | Tiền xử lý | Mỗi truy vấn | Space |
|---|---|---|---|
| 1. Quét chuỗi con | — | `O(n)` | `O(n)` (lát cắt) |
| 2. Tiền tố + nến gần nhất | `O(n)` | `O(1)` | `O(n)` |
| 3. Vị trí nến + `bisect` | `O(n)` | `O(log n)` | `O(#nến)` |
| 4. `cum` theo nến | `O(n)` | `O(1)` | `O(n)` |

In [13]:
rng = random.Random(2055)
n = 10 ** 5
s_big = "".join(rng.choice("**|") for _ in range(n))
q_big = [sorted((rng.randrange(n), rng.randrange(n))) for _ in range(10 ** 5)]
for name, f in [("prefix", plates_prefix), ("bisect", plates_bisect), ("submit", Solution().platesBetweenCandles)]:
    t = time.perf_counter(); r = f(s_big, q_big); print(f"{name:<7}: tổng {sum(r)} ({(time.perf_counter() - t) * 1000:.0f} ms)")
t = time.perf_counter(); plates_bruteforce(s_big, q_big[:2000]); print(f"brute 2000 truy vấn: {(time.perf_counter() - t) * 1000:.0f} ms (x50 cho 10^5)")

prefix : tổng 2223061431 (94 ms)


bisect : tổng 2223061431 (59 ms)
submit : tổng 2223061431 (56 ms)


brute 2000 truy vấn: 193 ms (x50 cho 10^5)


## 12. Bộ test

In [14]:
ALL = {"prefix": plates_prefix, "bisect": plates_bisect, "submit": Solution().platesBetweenCandles}
def run_all(s_, qs, brute=True):
    res = {k: f(s_, qs) for k, f in ALL.items()}
    if brute:
        res["brute"] = plates_bruteforce(s_, qs)
    vals = list(res.values())
    assert all(v == vals[0] for v in vals), (s_, qs, res)
    return vals[0]

for (s_, qs), exp in examples:
    assert run_all(s_, qs) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kiểm bằng quét trực tiếp trong run_all
edge = [(("***", [[0, 2]]), [0]), (("|||", [[0, 2]]), [0]), (("|*|", [[0, 2], [0, 1], [1, 2]]), [1, 0, 0]),
        (("*|*|*", [[0, 4], [1, 3], [2, 2]]), [1, 1, 0]), (("||**||**|*", [[3, 8]]), [2])]
for (s_, qs), exp in edge:
    assert run_all(s_, qs) == exp, (s_, qs, run_all(s_, qs))
rng = random.Random(20550)
for _ in range(400):
    n = rng.randint(3, 40); s_ = "".join(rng.choice(rng.choice(["*|", "**|", "*||"])) for _ in range(n))
    run_all(s_, [sorted((rng.randrange(n), rng.randrange(n))) for _ in range(30)])
for _ in range(3):
    n = 5000; s_ = "".join(rng.choice("*|") for _ in range(n))
    run_all(s_, [sorted((rng.randrange(n), rng.randrange(n))) for _ in range(2000)])
print("Ví dụ, 5 edge case, 400 chuỗi ngẫu nhiên, 3 chuỗi 5000 ký tự: OK")

Ví dụ, 5 edge case, 400 chuỗi ngẫu nhiên, 3 chuỗi 5000 ký tự: OK


### 12.1 Kiểm tra tính chất

- Đáp án `≤` số đĩa trong `s[l..r]`, và `= 0` khi đoạn có ít hơn 2 nến.
- Mở rộng đoạn (giảm `l` hoặc tăng `r`) không làm đáp án giảm.
- Đảo ngược chuỗi và đổi truy vấn `[l, r] → [n−1−r, n−1−l]` không đổi đáp án.

In [15]:
rng = random.Random(121)
f = Solution().platesBetweenCandles
for _ in range(300):
    n = rng.randint(3, 30); s_ = "".join(rng.choice("*|") for _ in range(n))
    qs = [sorted((rng.randrange(n), rng.randrange(n))) for _ in range(20)]
    a = f(s_, qs)
    for (l, r), x in zip(qs, a):
        assert x <= s_[l:r + 1].count("*") and (s_[l:r + 1].count("|") >= 2 or x == 0)
    wider = [[max(0, l - 1), min(n - 1, r + 1)] for l, r in qs]
    assert all(y >= x for x, y in zip(a, f(s_, wider)))
    assert f(s_[::-1], [[n - 1 - r, n - 1 - l] for l, r in qs]) == a
print("3 tính chất đúng trên 300 chuỗi")

3 tính chất đúng trên 300 chuỗi


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm mọi đĩa trong đoạn

In [16]:
def wrong_all_plates(s, queries):
    pre = list(accumulate((c == "*" for c in s), initial=0))
    return [pre[r + 1] - pre[l] for l, r in queries]                       # SAI

for (s_, qs), exp in examples:
    print("sai :", wrong_all_plates(s_, qs), "| đúng:", exp)

sai : [2, 3] | đúng: [2, 3]
sai : [12, 2, 2, 6, 0] | đúng: [9, 0, 0, 0, 0]


**Vì sao sai:** đĩa ở hai đầu đoạn (trước nến đầu / sau nến cuối) không "nằm giữa nến". **Sửa:** thu hẹp về `[a, b]`.

### Lỗi 2 — Lấy nến gần nhất **bên trái của `l`** (ra ngoài đoạn)

In [17]:
def wrong_outside(s, queries):
    n = len(s); pre = list(accumulate((c == "*" for c in s), initial=0)); nxt, prv = near_candles(s)
    out = []
    for l, r in queries:
        a, b = prv[l], prv[r]                                               # SAI: prv[l] có thể < l
        out.append(pre[b + 1] - pre[a] if a != -1 and a < b else 0)
    return out

for (s_, qs), exp in examples:
    print("sai :", wrong_outside(s_, qs), "| đúng:", exp)

sai : [2, 3] | đúng: [2, 3]
sai : [0, 0, 2, 2, 0] | đúng: [9, 0, 0, 0, 0]


**Vì sao sai:** nến bên trái phải nằm **trong** chuỗi con; `prv[l]` có thể ở trước `l` ⇒ đếm cả đĩa ngoài đoạn. **Sửa:** `a = nxt[l]` (nến đầu tiên `≥ l`).

### Lỗi 3 — Quên trường hợp `a ≥ b` (ra số âm)

In [18]:
def wrong_negative(s, queries):
    pre = list(accumulate((c == "*" for c in s), initial=0)); nxt, prv = near_candles(s)
    return [pre[prv[r] + 1] - pre[nxt[l]] for l, r in queries]             # SAI: không kiểm a < b

for (s_, qs), exp in examples:
    print("sai :", wrong_negative(s_, qs), "| đúng:", exp)

sai : [2, 3] | đúng: [2, 3]
sai : [9, -2, 0, 0, 0] | đúng: [9, 0, 0, 0, 0]


**Vì sao sai:** đoạn không có nến hoặc chỉ một nến cho `a ≥ b` ⇒ hiệu tiền tố âm hoặc đếm nhầm (thậm chí `nxt[l] = n` làm `pre[n]`). **Sửa:** `if a < b else 0`.

### Lỗi 4 — Lệch chỉ số tiền tố (`pre[b] − pre[a]` hay `pre[b] − pre[a+1]`)

In [19]:
def wrong_offset(s, queries):
    pre = list(accumulate((c == "*" for c in s), initial=0)); nxt, prv = near_candles(s)
    out = []
    for l, r in queries:
        a, b = nxt[l], prv[r]
        out.append(pre[b] - pre[a + 1] if a < b else 0)                     # "lệch" nhưng ...
    return out

rng = random.Random(4); bad = 0
for _ in range(500):
    n = rng.randint(3, 20); s_ = "".join(rng.choice("*|") for _ in range(n))
    qs = [sorted((rng.randrange(n), rng.randrange(n))) for _ in range(10)]
    bad += wrong_offset(s_, qs) != Solution().platesBetweenCandles(s_, qs)
print("pre[b] - pre[a+1] sai ở", bad, "/ 500 chuỗi -> thực ra luôn đúng vì s[a], s[b] là nến")

def wrong_offset2(s, queries):
    pre = list(accumulate((c == "*" for c in s), initial=0)); nxt, prv = near_candles(s)
    return [pre[r + 1] - pre[l] - (nxt[l] - l) if nxt[l] < prv[r] else 0 for l, r in queries]   # SAI: quên phần sau nến cuối

for (s_, qs), exp in examples:
    print("sai :", wrong_offset2(s_, qs), "| đúng:", exp)

pre[b] - pre[a+1] sai ở 0 / 500 chuỗi -> thực ra luôn đúng vì s[a], s[b] là nến
sai : [2, 3] | đúng: [2, 3]
sai : [10, 0, 1, 0, 0] | đúng: [9, 0, 0, 0, 0]


**Ghi chú:** `pre[b] − pre[a+1]` vẫn đúng vì `s[a]`, `s[b]` là nến (không phải đĩa) — một ví dụ "trông lệch nhưng đúng"; hãy hiểu **vì sao** đúng thay vì đoán. Lỗi thật ở bản thứ hai: chỉ trừ đĩa trước nến đầu mà quên trừ đĩa **sau nến cuối**. **Sửa:** dùng đúng `[a, b]`.

### Lỗi 5 — Quét từng truy vấn với dữ liệu lớn

In [20]:
s_long = "|" + "*" * 99998 + "|"
qs = [[0, 99999]] * 3000
for name, f in [("quét", plates_bruteforce), ("tiền tố", Solution().platesBetweenCandles)]:
    t = time.perf_counter(); f(s_long, qs); print(f"{name:<8}: 3000 truy vấn toàn chuỗi {(time.perf_counter() - t) * 1000:.0f} ms")

quét    : 3000 truy vấn toàn chuỗi 105 ms


tiền tố : 3000 truy vấn toàn chuỗi 14 ms


**Vì sao không tốt:** mỗi truy vấn cắt và đếm cả chuỗi `O(n)`; `10⁵` truy vấn như vậy là `10¹⁰`. **Sửa:** tiền xử lý.

### Lỗi 6 — `bisect` sai hàm (dùng `bisect_left` cho nến cuối `≤ r`)

In [21]:
def wrong_bisect(s, queries):
    pos = [i for i, c in enumerate(s) if c == "|"]; out = []
    for l, r in queries:
        i = bisect.bisect_left(pos, l); j = bisect.bisect_left(pos, r) - 1        # SAI khi s[r] là nến
        out.append(pos[j] - pos[i] - (j - i) if i < j else 0)
    return out

for (s_, qs), exp in examples:
    print("sai :", wrong_bisect(s_, qs), "| đúng:", exp)

sai : [0, 0] | đúng: [2, 3]
sai : [9, 0, 0, 0, 0] | đúng: [9, 0, 0, 0, 0]


**Vì sao sai:** nếu `s[r]` là nến, `bisect_left(pos, r)` trỏ **đúng** vào nó, trừ 1 thì bỏ mất nến đó. **Sửa:** `bisect_right(pos, r) − 1` cho "phần tử cuối `≤ r`".

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Nhiều truy vấn đoạn trên mảng tĩnh ⇒ **tiền xử lý** (tiền tố, gần nhất trái/phải) để mỗi truy vấn `O(1)`.
- "Phần tử nằm giữa hai mốc trong đoạn" ⇒ thu hẹp đoạn về **mốc đầu tiên ≥ l** và **mốc cuối cùng ≤ r**.
- `bisect_left(pos, l)` = đầu tiên `≥ l`; `bisect_right(pos, r) − 1` = cuối cùng `≤ r`.
- Luôn kiểm tra trường hợp rỗng (`a ≥ b`).

## 15. Biến thể của bài

1. **Đếm đĩa giữa hai nến kề nhau lớn nhất** trong đoạn — cần cấu trúc max đoạn (sparse table) trên khoảng cách giữa nến.
2. **Cập nhật động** (đổi ký tự) — Fenwick cho số đĩa + `SortedList` vị trí nến.
3. **303 Range Sum Query**, **1732**: tiền tố cơ bản.
4. **42 Trapping Rain Water**: "giữa hai mốc" với chiều cao.
5. **Đĩa giữa ít nhất `k` nến mỗi bên** — dịch chỉ số trong `pos` thêm `k − 1`.

### 15.1 Code biến thể: cập nhật động với `SortedList` + Fenwick

In [22]:
from sortedcontainers import SortedList

class DynamicPlates:
    def __init__(self, s):
        self.s = list(s); n = len(s); self.n = n; self.bit = [0] * (n + 1)
        self.candles = SortedList(i for i, c in enumerate(s) if c == "|")
        for i, c in enumerate(s):
            if c == "*": self._add(i, 1)
    def _add(self, i, d):
        i += 1
        while i <= self.n: self.bit[i] += d; i += i & -i
    def _sum(self, i):                        # số đĩa trong [0, i)
        r = 0
        while i > 0: r += self.bit[i]; i -= i & -i
        return r
    def set(self, i, c):
        if self.s[i] == c: return
        if c == "|": self.candles.add(i); self._add(i, -1)
        else: self.candles.remove(i); self._add(i, 1)
        self.s[i] = c
    def query(self, l, r):
        i = self.candles.bisect_left(l); j = self.candles.bisect_right(r) - 1
        if i >= j: return 0
        a, b = self.candles[i], self.candles[j]
        return self._sum(b + 1) - self._sum(a)

rng = random.Random(15)
for _ in range(100):
    n = rng.randint(3, 25); s_ = "".join(rng.choice("*|") for _ in range(n)); d = DynamicPlates(s_); cur = list(s_)
    for _ in range(30):
        if rng.random() < 0.4:
            i, c = rng.randrange(n), rng.choice("*|"); d.set(i, c); cur[i] = c
        else:
            l, r = sorted((rng.randrange(n), rng.randrange(n)))
            assert d.query(l, r) == by_definition("".join(cur), l, r)
print("Phiên bản cập nhật động (SortedList + Fenwick) khớp định nghĩa")

Phiên bản cập nhật động (SortedList + Fenwick) khớp định nghĩa


## 16. So sánh các cách làm

| Cách | Ý tưởng | Truy vấn | Khi nào dùng |
|---|---|---|---|
| 1. Quét | `find`/`rfind`/`count` | `O(n)` | Kiểm chứng |
| 2. Tiền tố + gần nhất | 3 mảng | `O(1)` | **Bản nộp** |
| 3. `bisect` vị trí nến | Công thức `pos[j]−pos[i]−(j−i)` | `O(log n)` | Ít bộ nhớ |
| 4. `cum` theo nến | Đĩa trước mỗi nến | `O(1)` | Biến thể của 2 |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `s = "*|*|*"`, truy vấn `[0,4]`, `[1,3]`, `[2,2]`, `[0,2]`.

**Bài 2 (tính tay):** Với `s = "||**||**|*"`, lập `nxt`, `prv`, `pre` và trả lời `[3, 8]`, `[0, 9]`, `[2, 3]`.

**Bài 3 (điền TODO):** Hoàn thiện `prev_candle(s)`.

**Bài 4 (tự viết lại):** Viết `plates_at_least_k(s, queries, k)`: đếm đĩa có ít nhất `k` nến bên trái **và** `k` nến bên phải trong đoạn.

**Bài 5 (mini challenge):** Viết `max_gap_between_candles(s, queries)`: với mỗi truy vấn, số đĩa **lớn nhất** giữa hai nến **kề nhau** trong đoạn (0 nếu ít hơn 2 nến), dùng sparse table trên mảng khoảng cách giữa các nến liên tiếp; đối chiếu vét cạn.

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def prev_candle(s):
    prv, last = [-1] * len(s), -1
    for i, c in enumerate(s):
        # TODO: nếu c là nến thì last = i; gán prv[i] = last
        pass
    return prv

In [24]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def plates_at_least_k(s, queries, k):
    # TODO: pos = vị trí nến; i = bisect_left(pos, l) + (k - 1); j = bisect_right(pos, r) - 1 - (k - 1)
    pass

def max_gap_between_candles(s, queries):
    # TODO: gap[t] = pos[t+1] - pos[t] - 1; sparse table max trên gap; truy vấn đoạn gap[i..j-1]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: nến ở 1, 3. Mức 2: đĩa giữa là vị trí 2. Mức 3: `[1, 1, 0, 0]`.

**Bài 2.** Mức 1: nến ở 0, 1, 4, 5, 8. Mức 2: `[3,8]`: `a = 4`, `b = 8` ⇒ 2. Mức 3: `[0,9]`: `a = 0`, `b = 8` ⇒ 4; `[2,3]`: không có nến ⇒ 0.

**Bài 3.** Mức 1: duyệt trái → phải. Mức 2: cập nhật `last`. Mức 3: gán sau khi cập nhật (nến tự là nến gần nhất của chính nó).

**Bài 4.** Mức 1: đĩa ở giữa nến thứ `k` từ trái và nến thứ `k` từ phải của đoạn. Mức 2: dịch `i`, `j` đi `k − 1`. Mức 3: công thức Cách 3 với `i < j`.

**Bài 5.** Mức 1: khoảng cách giữa nến kề nhau là một mảng `gap`. Mức 2: truy vấn = max `gap[i..j−1]`. Mức 3: sparse table `O(m log m)` tiền xử lý, `O(1)` truy vấn.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2, 3
f = Solution().platesBetweenCandles
b1 = f("*|*|*", [[0, 4], [1, 3], [2, 2], [0, 2]])
print("Bài 1:", b1); assert b1 == [1, 1, 0, 0]
b2 = f("||**||**|*", [[3, 8], [0, 9], [2, 3]])
print("Bài 2:", b2); assert b2 == [2, 4, 0]
def prev_candle(s):
    prv, last = [-1] * len(s), -1
    for i, c in enumerate(s):
        if c == "|": last = i
        prv[i] = last
    return prv
assert prev_candle("**|**|***|") == near_candles("**|**|***|")[1]
print("Bài 3: OK")

Bài 1: [1, 1, 0, 0]
Bài 2: [2, 4, 0]
Bài 3: OK


In [26]:
# Bài 4, 5
def plates_at_least_k(s, queries, k):
    pos = [i for i, c in enumerate(s) if c == "|"]; out = []
    for l, r in queries:
        i = bisect.bisect_left(pos, l) + (k - 1); j = bisect.bisect_right(pos, r) - 1 - (k - 1)
        out.append(pos[j] - pos[i] - (j - i) if i < j else 0)
    return out

def max_gap_between_candles(s, queries):
    pos = [i for i, c in enumerate(s) if c == "|"]
    gap = [pos[t + 1] - pos[t] - 1 for t in range(len(pos) - 1)]
    m = len(gap); sp = [gap[:]]
    k = 1
    while (1 << k) <= m:
        prev = sp[-1]; half = 1 << (k - 1)
        sp.append([max(prev[i], prev[i + half]) for i in range(m - (1 << k) + 1)]); k += 1
    out = []
    for l, r in queries:
        i = bisect.bisect_left(pos, l); j = bisect.bisect_right(pos, r) - 1
        if i >= j: out.append(0); continue
        a, b = i, j - 1; lv = (b - a + 1).bit_length() - 1
        out.append(max(sp[lv][a], sp[lv][b - (1 << lv) + 1]))
    return out

rng = random.Random(5)
for _ in range(300):
    n = rng.randint(3, 25); s_ = "".join(rng.choice("*|") for _ in range(n))
    qs = [sorted((rng.randrange(n), rng.randrange(n))) for _ in range(10)]; k = rng.randint(1, 3)
    exp_k = [sum(1 for x in range(l, r + 1) if s_[x] == "*" and s_[l:x].count("|") >= k and s_[x + 1:r + 1].count("|") >= k) for l, r in qs]
    assert plates_at_least_k(s_, qs, k) == exp_k
    exp_g = []
    for l, r in qs:
        ps = [i for i in range(l, r + 1) if s_[i] == "|"]
        exp_g.append(max((ps[t + 1] - ps[t] - 1 for t in range(len(ps) - 1)), default=0))
    assert max_gap_between_candles(s_, qs) == exp_g
print("Bài 4, 5: OK |", plates_at_least_k("|*|**|***|", [[0, 9]], 2), max_gap_between_candles("|*|**|***|", [[0, 9], [0, 5]]))

Bài 4, 5: OK | [2] [3, 2]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
pre[i+1] = pre[i] + (s[i] == '*')                    # tiền tố đĩa
nxt[i] = i nếu s[i] là nến, ngược lại nxt[i+1]        # duyệt phải -> trái
prv[i] = i nếu s[i] là nến, ngược lại prv[i-1]        # duyệt trái -> phải
for l, r in queries:
    a, b = nxt[l], prv[r]
    ans = pre[b + 1] - pre[a] if a < b else 0
```

**Checklist:**
- [ ] Nến đầu là `nxt[l]` (trong đoạn), nến cuối là `prv[r]`?
- [ ] Kiểm tra `a < b` (0 hoặc 1 nến ⇒ 0)?
- [ ] Chỉ số tiền tố đúng `pre[b+1] − pre[a]`?
- [ ] `bisect_right(pos, r) − 1` cho nến cuối `≤ r`?
- [ ] Tiền xử lý `O(n)`, truy vấn `O(1)`/`O(log n)`?